# Family B - EfficientNetV2-S: augmentation study + 5 folds + test ensemble

`tf_efficientnetv2_s.in21k_ft_in1k` (pure CNN: MBConv + squeeze-excitation) fine-tuned on the team's
shared clean split (Elia's `folds.csv`, 2,995 images) and evaluated on the clean and the **shifted**
(test-like) validation (`val_shift_plan.csv`). Full study and reasons: `families/B_effnetv2/DECISIONS.md`.

**What the study found (short)**
- The test differs from the clean training set by a **hue rotation of about ±120°** (~28% of the images)
  and a **mild blur** (~20%): both are reproduced by the shifted validation.
- Two more differences are real but **not** in the shifted validation: more off-centre galaxies and a
  wider sky-brightness range. (The galaxies only *look* larger in the test because of the blur.)
- A ±120° hue rotation is exactly a cyclic shift of the RGB channels -> cheap hue-view TTA.
- Reference: Elia's ConvNeXt-Tiny (flips only) OOF **0.844 clean / 0.820 shifted**, Kaggle public **0.832**.

**What the notebook does**
1. **Stage 1 - augmentation study on fold 0** (no test prediction): control = any-angle rotation + flip,
   then adds hue rotation, blur, photometric + translation one at a time, and keeps each step only if
   it passes the team rule (shifted F1 +≥0.01, clean −≤0.01; "no harm" for the photometric step, which
   targets shifts the validation cannot see). Decisions are printed with their numbers.
2. **Stage 2 - 5 folds** with the chosen augmentation (fold 0 is reused from stage 1). Each fold model
   predicts the test with D4 TTA, also on the two channel-shifted views (hue-view TTA).
3. **OOF + submissions**: OOF scores for plain D4 and two hue-TTA weightings; `submission.csv` uses the
   mode chosen by the team rule, and both variants are saved (the shifted validation flatters hue-TTA,
   so the Kaggle public score decides).

**Kaggle settings**: *Add Input* `an2dl2627-data` + `batchsize4-shared-split`; *Accelerator* **GPU T4 x2**
(one GPU is used); *Internet* **on**. Check: `SMOKE = True` + *Run All* (~3 min). Full run:
`SMOKE = False`, *Save Version > Save & Run All* (runs on Kaggle's servers, the browser can be closed).
Outputs: `/kaggle/working/outputs/<RUN>/` and `/kaggle/working/submission.csv`.

In [ ]:
# ---- configuration ----
MODEL = 'tf_efficientnetv2_s.in21k_ft_in1k'   # in21k weights: clearly stronger than the in1k ones
SMOKE = False            # True = tiny run (40 images per fold, 1 epoch): checks the pipeline only

RUN_ABLATION = True      # stage 1: augmentation study on ABLATION_FOLD
ABLATION_FOLD = 0
AUG_FIXED = None         # e.g. dict(hue_p=0.5, blur_p=0.3) to skip stage 1 and use this augmentation
RUN_FINAL = True         # stage 2: all FOLDS with the chosen augmentation
FOLDS = [0, 1, 2, 3, 4]
PREDICT_TEST = True      # every fold model predicts the test set -> averaged submission
HUE_TTA = True           # also predict the two channel-shifted views (= +-120 deg hue)

IMG_SIZE = 256           # native size
EPOCHS = 30
BATCH_SIZE = 32
LR = 3e-4                # AdamW, 1 epoch warmup then cosine
WEIGHT_DECAY = 0.05
LABEL_SMOOTHING = 0.1
DROP_RATE, DROP_PATH_RATE = 0.2, 0.1
EMA_DECAY = 0.998        # final weights = EMA of the last epoch, no best-epoch picking
EVAL_EVERY = 2           # quick validation (EMA, no TTA) every N epochs
SEED = 42
NUM_WORKERS = 4          # only for decoding the test PNGs

# augmentation (GPU, one random draw per image). Rotation by any angle + horizontal flip is always on.
AUG_BASE = dict(hue_p=0.0, hue_max_turn=0.5,          # HSV hue rotation, 0.5 turn = up to +-180 deg
                blur_p=0.0, blur_sigma=(0.3, 1.3),    # Gaussian blur, sigma in px
                photometric=False,                    # brightness / contrast / sky offset / light noise
                brightness=(0.8, 1.15), contrast=(0.85, 1.15), sky_offset=(-0.03, 0.04),
                noise_p=0.2, noise_sigma=(0.0, 0.015),
                translate=0.0)                        # max shift, fraction of the image side
# stage 1 adds these one at a time (rule 'improve' = team rule, 'no_harm' = the validation cannot see it)
AUG_STEPS = [
    ('hue rotation', dict(hue_p=0.5), 'improve'),
    ('blur', dict(blur_p=0.3), 'improve'),
    ('photometric + translation', dict(photometric=True, translate=0.04), 'no_harm'),
]
RUN = 'B_effnetv2s' + ('_smoke' if SMOKE else '')

In [ ]:
import glob, os, sys, time, json, random, shutil, math, gc
from pathlib import Path

# common.py comes with Elia's dataset (batchsize4-shared-split); off Kaggle set AN2DL_COMMON_DIR
_hits = glob.glob('/kaggle/input/**/common.py', recursive=True)
COMMON_DIR = os.path.dirname(_hits[0]) if _hits else os.environ['AN2DL_COMMON_DIR']
sys.path.insert(0, COMMON_DIR)
import common as C

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import timm
import timm.utils
from PIL import Image

DEVICE = torch.device('cuda' if torch.cuda.is_available() else
                      'mps' if torch.backends.mps.is_available() else 'cpu')
USE_AMP = DEVICE.type == 'cuda'
# bf16 only on GPUs that have it in hardware (A100/L4); T4/P100 would emulate it slowly -> fp16
AMP_DTYPE = torch.bfloat16 if USE_AMP and torch.cuda.get_device_capability()[0] >= 8 else torch.float16
torch.backends.cudnn.benchmark = True

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

run_dir = C.OUT / RUN
run_dir.mkdir(parents=True, exist_ok=True)
T_START = time.time()
print('common.py:', COMMON_DIR)
print('DATA  =', C.DATA)
print('OUT   =', run_dir)
print('device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else '',
      '| amp:', AMP_DTYPE if USE_AMP else 'off', '| torch', torch.__version__, '| timm', timm.__version__)

## Input checks

In [ ]:
train_csv = pd.read_csv(C.DATA / 'train.csv')
sample_sub = pd.read_csv(C.DATA / 'sample_submission.csv')
folds = pd.read_csv(C.SPLITS / 'folds.csv')
plan = pd.read_csv(C.SPLITS / 'val_shift_plan.csv').set_index('id')
n_test_png = len(list((C.DATA / 'test').glob('*.png')))

assert len(train_csv) == 4328, len(train_csv)
assert n_test_png == 14736 and len(sample_sub) == 14736, (n_test_png, len(sample_sub))
assert list(sample_sub.columns) == ['id', 'label']
assert len(folds) == 2995 and sorted(folds.fold.unique()) == [0, 1, 2, 3, 4]
assert plan.index.tolist() == folds.id.tolist(), 'shift plan not aligned with folds.csv'
assert all((C.DATA / 'train' / f'{i}.png').exists() for i in folds.id), 'missing train images'
lab = train_csv.set_index('id').label
assert (lab[folds.id].values == folds.label.values).all(), 'labels differ from train.csv'
print('inputs OK: 4,328 train rows, 2,995 clean images in 5 folds, 14,736 test images')

## Images: clean set (+ shifted copy) on the GPU, test set in RAM

In [ ]:
if SMOKE:
    FOLDS, EPOCHS, BATCH_SIZE, EVAL_EVERY = FOLDS[:2], 1, 8, 1    # batch 8 fits an 8 GB Mac
    folds = folds.groupby('fold').head(40).reset_index(drop=True)
    plan = plan.loc[folds.id]
te_ids = sample_sub.id.tolist()[:64] if SMOKE else sample_sub.id.tolist()

def to_tensor(images):
    arr = np.stack([np.asarray(im, dtype=np.uint8) for im in images])
    return torch.from_numpy(arr).permute(0, 3, 1, 2).contiguous()     # uint8 (N, 3, H, W)

t0 = time.time()
X_all = to_tensor(C.load_image(i) for i in folds.id).to(DEVICE)
X_all_shift = to_tensor(C.load_shifted(i, plan) for i in folds.id).to(DEVICE)   # the team's fixed shifted copy
y_all = folds.label.values
y_all_t = torch.as_tensor(y_all, device=DEVICE)
print(f'clean set on {DEVICE}: {len(X_all)} images (+ shifted copy) in {time.time() - t0:.0f}s')

class PngDataset(torch.utils.data.Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        return torch.from_numpy(np.asarray(Image.open(self.paths[i]).convert('RGB')).copy()).permute(2, 0, 1)

X_test = None
if RUN_FINAL and PREDICT_TEST:
    t0 = time.time()
    workers = NUM_WORKERS if sys.platform.startswith('linux') else 0   # macOS can't pickle notebook classes
    dl = torch.utils.data.DataLoader(PngDataset([C.DATA / 'test' / f'{i}.png' for i in te_ids]),
                                     batch_size=256, num_workers=workers)
    X_test = torch.cat(list(dl))                                       # uint8, ~2.9 GB of RAM
    print(f'test set in RAM: {tuple(X_test.shape)} in {time.time() - t0:.0f}s')

## GPU augmentation

In [ ]:
# float images in [0, 1], shape (B, 3, H, W), one random parameter per image
def rgb_to_hsv(x):
    r, g, b = x.unbind(1)
    maxc, minc = x.max(1)[0], x.min(1)[0]
    delta = maxc - minc
    d = delta.clamp_min(1e-8)
    h = torch.where(maxc == r, (g - b) / d, torch.where(maxc == g, 2.0 + (b - r) / d, 4.0 + (r - g) / d))
    h = torch.where(delta > 0, (h / 6.0) % 1.0, torch.zeros_like(h))
    s = torch.where(maxc > 0, delta / maxc.clamp_min(1e-8), torch.zeros_like(maxc))
    return torch.stack([h, s, maxc], 1)

def hsv_to_rgb(x):
    h, s, v = x.unbind(1)
    i = torch.floor(h * 6.0)
    f = h * 6.0 - i
    p, q, t = v * (1 - s), v * (1 - s * f), v * (1 - s * (1 - f))
    i = (i.long() % 6).unsqueeze(1)
    r = torch.stack([v, q, p, p, t, v], 1).gather(1, i)
    g = torch.stack([t, v, v, q, p, p], 1).gather(1, i)
    b = torch.stack([p, p, t, v, v, q], 1).gather(1, i)
    return torch.cat([r, g, b], 1)

def hue_rotate(x, turns):
    hsv = rgb_to_hsv(x)
    hsv[:, 0] = (hsv[:, 0] + turns.view(-1, 1, 1)) % 1.0
    return hsv_to_rgb(hsv).clamp(0, 1)

def gaussian_blur(x, sigma, ksize=9):
    B, Ch, H, W = x.shape
    half = ksize // 2
    coords = torch.arange(-half, half + 1, device=x.device, dtype=x.dtype)
    k = torch.exp(-coords.view(1, -1) ** 2 / (2 * sigma.to(x.dtype).clamp_min(1e-3).view(-1, 1) ** 2))
    k = (k / k.sum(1, keepdim=True)).repeat_interleave(Ch, 0)
    y = x.reshape(1, B * Ch, H, W)
    y = F.conv2d(F.pad(y, (half, half, 0, 0), mode='reflect'), k.view(B * Ch, 1, 1, -1), groups=B * Ch)
    y = F.conv2d(F.pad(y, (0, 0, half, half), mode='reflect'), k.view(B * Ch, 1, -1, 1), groups=B * Ch)
    return y.view(B, Ch, H, W)

def affine(x, angle, flip, tx, ty):
    # rotation by any angle, optional horizontal flip, translation (normalised units); reflection padding
    cos, sin = torch.cos(angle), torch.sin(angle)
    theta = torch.zeros(x.shape[0], 2, 3, device=x.device, dtype=x.dtype)
    theta[:, 0, 0], theta[:, 0, 1], theta[:, 0, 2] = cos * flip, -sin, tx
    theta[:, 1, 0], theta[:, 1, 1], theta[:, 1, 2] = sin * flip, cos, ty
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    return F.grid_sample(x, grid, mode='bilinear', padding_mode='reflection', align_corners=False)

def on_subset(x, mask, fn, *params):
    if mask.any():
        x = x.clone()
        x[mask] = fn(x[mask], *[p[mask] for p in params])
    return x

def augment(x, A):
    B = x.shape[0]
    u = lambda lo=0.0, hi=1.0: (torch.rand(B, generator=GEN) * (hi - lo) + lo).to(x.device)
    if A['hue_p'] > 0:
        x = on_subset(x, u() < A['hue_p'], hue_rotate, u(-A['hue_max_turn'], A['hue_max_turn']))
    if A['blur_p'] > 0:
        x = on_subset(x, u() < A['blur_p'], gaussian_blur, u(*A['blur_sigma']))
    if A['photometric']:
        b, c, o = u(*A['brightness']), u(*A['contrast']), u(*A['sky_offset'])
        mean = x.mean(dim=(1, 2, 3), keepdim=True)
        x = ((x - mean) * c.view(-1, 1, 1, 1) + mean) * b.view(-1, 1, 1, 1) + o.view(-1, 1, 1, 1)
        sigma = (u() < A['noise_p']).float() * u(*A['noise_sigma'])
        x = (x + torch.randn_like(x) * sigma.view(-1, 1, 1, 1)).clamp(0, 1)
    flip = torch.where(u() < 0.5, -1.0, 1.0)
    t = 2 * A['translate']                                  # fraction of the side -> normalised units
    return affine(x, u(0, 2 * math.pi), flip, u(-t, t), u(-t, t))

def prepare(x_uint8, A=None, hue_shift=0):
    # uint8 batch (any device) -> normalised float on the GPU; augmented if A is given.
    # hue_shift: +1 / -1 rolls the RGB channels = exact +120 / -120 deg hue rotation (done on [0,1] pixels)
    x = x_uint8.to(DEVICE, non_blocking=True).float().div_(255)
    if A is not None:
        x = augment(x, A)
    if hue_shift:
        x = x.roll(hue_shift, dims=1)
    if IMG_SIZE != x.shape[-1]:
        x = F.interpolate(x, size=(IMG_SIZE, IMG_SIZE), mode='bilinear', antialias=True, align_corners=False)
    return ((x - MEAN_T) / STD_T).contiguous(memory_format=torch.channels_last)

## Model, training, evaluation

In [ ]:
_cfg = timm.create_model(MODEL, pretrained=False).pretrained_cfg
MEAN_T = torch.tensor(_cfg['mean'], device=DEVICE).view(1, 3, 1, 1)
STD_T = torch.tensor(_cfg['std'], device=DEVICE).view(1, 3, 1, 1)
HUE_VIEWS = [0, 1, -1] if HUE_TTA else [0]     # original, +120 deg, -120 deg

def make_model(pretrained=True):
    net = timm.create_model(MODEL, pretrained=pretrained, num_classes=C.N_CLASSES,
                            drop_rate=DROP_RATE, drop_path_rate=DROP_PATH_RATE)
    head = net.get_classifier()          # zero-init the new head: starts at uniform predictions
    nn.init.zeros_(head.weight); nn.init.zeros_(head.bias)
    return net.to(DEVICE).to(memory_format=torch.channels_last)

@torch.no_grad()
def predict(net, X, d4=False, hue_shift=0, bs=None):
    # logits (N, 10) for a uint8 tensor (GPU or CPU); d4 = mean over 4 rotations x 2 flips
    net.eval()
    bs = bs or 2 * BATCH_SIZE
    out = []
    for s in range(0, len(X), bs):
        x = prepare(X[s:s + bs], hue_shift=hue_shift)
        views = [torch.rot90(v, k, (2, 3)) for v in ([x, x.flip(3)] if d4 else [x]) for k in (range(4) if d4 else [0])]
        acc = 0
        for v in views:
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                acc = acc + net(v).float()
        out.append((acc / len(views)).cpu())
    return torch.cat(out).numpy()

def scores(y, logits_clean, logits_shift):
    rc, rs = C.evaluate(y, logits_clean.argmax(1)), C.evaluate(y, logits_shift.argmax(1))
    return {'f1_clean': round(float(rc['f1_weighted']), 4), 'f1_shifted': round(float(rs['f1_weighted']), 4),
            'f1_cigar_clean': round(float(rc['f1_per_class'].iloc[4]), 4),
            'f1_cigar_shifted': round(float(rs['f1_per_class'].iloc[4]), 4)}

def log(row):
    print(' '.join(f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}' for k, v in row.items()), flush=True)

def fit(k, A, tag):
    # trains on folds != k with augmentation A, evaluates fold k (D4 TTA, every hue view), saves everything.
    # If tag was already trained in this session, reloads it. Returns (eval model, metrics).
    if (run_dir / f'{tag}_metrics.json').exists():
        net = make_model(pretrained=False)
        net.load_state_dict(torch.load(run_dir / f'{tag}_ema.pt', map_location=DEVICE))
        print(f'[{tag}] already trained, reloaded')
        return net, json.load(open(run_dir / f'{tag}_metrics.json'))
    seed_everything(SEED + k)
    global GEN
    GEN = torch.Generator().manual_seed(SEED * 100 + k)    # augmentation draws: reproducible per fold
    tr = np.where(folds.fold.values != k)[0]
    va = np.where(folds.fold.values == k)[0]
    tr_t = torch.as_tensor(tr, device=DEVICE)
    print(f'\n===== [{tag}] fold {k}: train {len(tr)} | val {len(va)} =====', flush=True)

    model = make_model()
    steps_per_epoch = len(tr) // BATCH_SIZE
    steps = EPOCHS * steps_per_epoch
    warmup = min(steps_per_epoch, steps // 10 + 1)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    scheduler = torch.optim.lr_scheduler.LambdaLR(
        optimizer, lambda s: (s + 1) / warmup if s < warmup else
        0.01 + 0.99 * 0.5 * (1 + np.cos(np.pi * (s - warmup) / max(1, steps - warmup))))
    criterion = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP and AMP_DTYPE == torch.float16)
    if hasattr(timm.utils, 'ModelEmaV3'):   # warmup: the average does not drag the random-head start along
        ema, ema_warmup = timm.utils.ModelEmaV3(model, decay=EMA_DECAY, use_warmup=True), True
    else:
        ema, ema_warmup = timm.utils.ModelEmaV2(model, decay=EMA_DECAY), False

    history, t0, step = [], time.time(), 0
    for epoch in range(1, EPOCHS + 1):
        model.train()
        te, tot, n = time.time(), 0.0, 0
        perm = tr_t[torch.randperm(len(tr), generator=GEN).to(DEVICE)]
        for s in range(steps_per_epoch):
            idx = perm[s * BATCH_SIZE:(s + 1) * BATCH_SIZE]
            x, y = prepare(X_all[idx], A=A), y_all_t[idx]
            with torch.autocast(DEVICE.type, dtype=AMP_DTYPE, enabled=USE_AMP):
                loss = criterion(model(x), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update(); scheduler.step()
            step += 1
            ema.update(model, step=step) if ema_warmup else ema.update(model)
            tot += loss.item() * len(y); n += len(y)
        row = {'tag': tag, 'epoch': epoch, 'train_loss': round(tot / n, 4),
               'lr': optimizer.param_groups[0]['lr'], 'sec': round(time.time() - te, 1)}
        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            lc, ls = predict(ema.module, X_all[va]), predict(ema.module, X_all_shift[va])
            row.update(val_loss=round(F.cross_entropy(torch.from_numpy(lc), torch.from_numpy(y_all[va])).item(), 4),
                       **{kk: v for kk, v in scores(y_all[va], lc, ls).items() if kk in ('f1_clean', 'f1_shifted')})
        history.append(row); log(row)

    net = ema.module
    del model, optimizer, ema
    # final: EMA weights of the last epoch, D4 TTA, every hue view (view 0 = plain image)
    clean = np.stack([predict(net, X_all[va], d4=True, hue_shift=v) for v in HUE_VIEWS])
    shift = np.stack([predict(net, X_all_shift[va], d4=True, hue_shift=v) for v in HUE_VIEWS])
    np.savez(run_dir / f'{tag}_val.npz', idx=va, clean=clean.astype(np.float32), shifted=shift.astype(np.float32))
    pd.DataFrame(history).to_csv(run_dir / f'{tag}_history.csv', index=False)
    torch.save(net.state_dict(), run_dir / f'{tag}_ema.pt')
    metrics = {'tag': tag, 'fold': k, **scores(y_all[va], clean[0], shift[0]),
               'minutes': round((time.time() - t0) / 60, 1), 'aug': A}
    json.dump(metrics, open(run_dir / f'{tag}_metrics.json', 'w'), indent=2)
    print(f'[{tag}] D4 TTA: f1_clean={metrics["f1_clean"]} f1_shifted={metrics["f1_shifted"]} '
          f'cigar {metrics["f1_cigar_clean"]}/{metrics["f1_cigar_shifted"]} | {metrics["minutes"]} min', flush=True)
    return net, metrics

def free():
    # call after `del net`: releases the GPU memory of models no longer referenced
    gc.collect()
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

## Stage 1 - augmentation study (fold 0)

Greedy, one change at a time, decided by the team rule (Elia's README): the **shifted** F1 decides
(gain ≥ 0.01), the clean F1 may drop at most 0.01, class 4 must not collapse. The photometric step
targets shifts that the shifted validation does not contain, so it only has to do no harm.
Differences below ~0.01 on a single fold are within noise: in that case the simpler recipe wins.

In [ ]:
ablation, WINNER_TAG = [], None
if AUG_FIXED is not None:
    AUG = {**AUG_BASE, **AUG_FIXED}
    print('stage 1 skipped, fixed augmentation:', AUG)
elif RUN_ABLATION:
    current = dict(AUG_BASE)
    net, best = fit(ABLATION_FOLD, current, 'abl0'); del net; free()
    best_tag = 'abl0'
    ablation.append({'step': 'control (rotation + flip)', **{k: best[k] for k in ('f1_clean', 'f1_shifted', 'f1_cigar_clean', 'f1_cigar_shifted', 'minutes')}, 'kept': True})
    for i, (name, change, rule) in enumerate(AUG_STEPS, 1):
        cand = {**current, **change}
        net, m = fit(ABLATION_FOLD, cand, f'abl{i}'); del net; free()
        d_clean, d_shift = m['f1_clean'] - best['f1_clean'], m['f1_shifted'] - best['f1_shifted']
        collapse = m['f1_cigar_shifted'] < 0.5 * best['f1_cigar_shifted']
        need = 0.01 if rule == 'improve' else -0.01
        kept = d_shift >= need and d_clean >= -0.01 and not collapse
        why = (f"shifted {d_shift:+.4f} (needs {'>=' if rule == 'improve' else '>='} {need:+.2f}), "
               f"clean {d_clean:+.4f} (needs >= -0.01){', class 4 collapsed' if collapse else ''}")
        print(f'\n>>> step "{name}": {"KEPT" if kept else "rejected"} - {why}\n', flush=True)
        ablation.append({'step': f'+ {name}', **{k: m[k] for k in ('f1_clean', 'f1_shifted', 'f1_cigar_clean', 'f1_cigar_shifted', 'minutes')}, 'kept': kept})
        if kept:
            current, best, best_tag = cand, m, f'abl{i}'
    AUG, WINNER_TAG = current, best_tag
else:
    AUG = dict(AUG_BASE)
    for _, change, _ in AUG_STEPS:
        AUG.update(change)
    print('stage 1 skipped, every step on:', AUG)

if ablation:
    abl = pd.DataFrame(ablation)
    abl.to_csv(run_dir / 'ablation.csv', index=False)
    print(abl.to_string(index=False))
    if SMOKE:
        print('(SMOKE: these numbers mean nothing, the run only checks the pipeline)')
json.dump(AUG, open(run_dir / 'aug_selected.json', 'w'), indent=2)
print('\nchosen augmentation:', {k: v for k, v in AUG.items() if v not in (0.0, False)}, '| from', WINNER_TAG)

## Stage 2 - all folds with the chosen augmentation

Fold 0 reuses the winning model of stage 1 (same data, seed and recipe). Each fold model predicts the
test with D4 TTA on every hue view; the raw logits are saved per fold.

In [ ]:
fold_metrics, final_tags = [], {}
if RUN_FINAL:
    for k in FOLDS:
        tag = WINNER_TAG if (k == ABLATION_FOLD and WINNER_TAG) else f'fold{k}'
        net, m = fit(k, AUG, tag)
        final_tags[k] = tag
        fold_metrics.append(m)
        tpath = run_dir / f'{tag}_test.npy'
        if PREDICT_TEST and not tpath.exists():
            tt = time.time()
            np.save(tpath, np.stack([predict(net, X_test, d4=True, hue_shift=v) for v in HUE_VIEWS]).astype(np.float32))
            print(f'[{tag}] test predicted ({len(HUE_VIEWS)} hue views x D4) in {(time.time() - tt) / 60:.1f} min', flush=True)
        del net; free()
    print(pd.DataFrame(fold_metrics)[['tag', 'fold', 'f1_clean', 'f1_shifted', 'f1_cigar_clean', 'f1_cigar_shifted', 'minutes']].to_string(index=False))
    print(f'\nelapsed so far: {(time.time() - T_START) / 60:.0f} min')

## OOF scores, TTA mode, submissions, team files

Three ways to combine the hue views (probabilities averaged): plain D4 (original view only), uniform
over the 3 views, and weighted 0.5 / 0.25 / 0.25. The team rule picks the mode used for
`submission.csv`. The shifted validation uses the same hue mechanism as the test generator we
assumed, so it favours hue-TTA: both submissions are written and the public score decides.

In [ ]:
def softmax(z):
    z = z - z.max(-1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(-1, keepdims=True)

if RUN_FINAL and fold_metrics:
    V = len(HUE_VIEWS)
    oof_c = np.full((V, len(folds), C.N_CLASSES), np.nan, np.float32)
    oof_s = oof_c.copy()
    for k, tag in final_tags.items():
        d = np.load(run_dir / f'{tag}_val.npz')
        oof_c[:, d['idx']], oof_s[:, d['idx']] = d['clean'], d['shifted']
    have = ~np.isnan(oof_c[0, :, 0])
    modes = {'D4': [1.0] + [0.0] * (V - 1)}
    if V == 3:
        modes['D4 + hue x3'] = [1 / 3] * 3
        modes['D4 + hue weighted'] = [0.5, 0.25, 0.25]
    combine = lambda L, w: np.tensordot(np.asarray(w), softmax(L), axes=1)     # (V, N, 10) -> (N, 10) probs
    res = {}
    for name, w in modes.items():
        pc, ps = combine(oof_c, w), combine(oof_s, w)
        res[name] = {'probs_clean': pc, 'probs_shifted': ps, **scores(y_all[have], pc[have], ps[have])}
    tab = pd.DataFrame({n: {k: v for k, v in r.items() if k.startswith('f1')} for n, r in res.items()}).T
    print('OOF over folds', sorted(final_tags), '\n', tab.to_string())

    chosen = 'D4'
    for name in sorted([n for n in modes if n != 'D4'], key=lambda n: -res[n]['f1_shifted']):
        if res[name]['f1_shifted'] - res['D4']['f1_shifted'] >= 0.01 and res[name]['f1_clean'] - res['D4']['f1_clean'] >= -0.01:
            chosen = name
            break
    best_hue = max([n for n in modes if n != 'D4'], key=lambda n: res[n]['f1_shifted'], default=None)
    print(f'\nTTA mode for submission.csv: {chosen}  (best hue mode on shifted OOF: {best_hue})')

    rc = C.evaluate(y_all[have], res[chosen]['probs_clean'][have].argmax(1))
    rs = C.evaluate(y_all[have], res[chosen]['probs_shifted'][have].argmax(1))
    print('\nper-class F1 (OOF, chosen mode):')
    print(pd.DataFrame({'clean': rc['f1_per_class'], 'shifted': rs['f1_per_class']}).round(3).to_string())

    # team files, Elia's format: oof.csv with probabilities, test_probs.npy (folds, N, 10)
    oof = pd.DataFrame({'id': folds.id, 'label': y_all, 'fold': folds.fold})
    for nm, P in [('clean', res[chosen]['probs_clean']), ('shifted', res[chosen]['probs_shifted'])]:
        for c in range(C.N_CLASSES):
            oof[f'p{c}_{nm}'] = P[:, c]
    oof.to_csv(run_dir / 'oof.csv', index=False)
    np.save(run_dir / 'oof_logits_views_clean.npy', oof_c)
    np.save(run_dir / 'oof_logits_views_shifted.npy', oof_s)

    slug = lambda n: n.replace(' + ', '_').replace(' ', '_')
    if PREDICT_TEST:
        T = np.stack([np.load(run_dir / f'{final_tags[k]}_test.npy') for k in sorted(final_tags)])   # (K, V, N, 10)
        written = []
        for name in dict.fromkeys(['D4', best_hue, chosen]):
            if name is None:
                continue
            P = np.stack([combine(T[f], modes[name]) for f in range(len(T))])          # (K, N, 10)
            np.save(run_dir / f'test_probs_{slug(name)}.npy', P.astype(np.float32))
            sub = sample_sub.iloc[:len(te_ids)].copy()
            assert sub.id.tolist() == te_ids
            sub['label'] = P.mean(0).argmax(1)
            sub.to_csv(run_dir / f'submission_{slug(name)}.csv', index=False)
            written.append(f'submission_{slug(name)}.csv')
            print(f'{name:18s} test class shares (%):', (100 * np.bincount(sub.label, minlength=10) / len(sub)).round(1).tolist())
        shutil.copy(run_dir / f'test_probs_{slug(chosen)}.npy', run_dir / 'test_probs.npy')
        if Path('/kaggle/working').exists() and not SMOKE:
            shutil.copy(run_dir / f'submission_{slug(chosen)}.csv', '/kaggle/working/submission.csv')
        print('\nwritten:', written, '| /kaggle/working/submission.csv =', f'submission_{slug(chosen)}.csv')

    # experiment log rows in Elia's columns
    now = pd.Timestamp.now().strftime('%Y-%m-%d %H:%M')
    rows = [{'date': now, 'fold': ABLATION_FOLD, 'train_set': 'clean (keep=True)', 'experiment': f'{RUN}_ablation',
             'kaggle': None, 'note': f"{a['step']} ({'kept' if a['kept'] else 'rejected'})",
             **{k: a[k] for k in ('f1_clean', 'f1_shifted', 'f1_cigar_clean', 'f1_cigar_shifted', 'minutes')}} for a in ablation]
    rows += [{'date': now, 'fold': m['fold'], 'train_set': 'clean (keep=True)', 'experiment': RUN, 'kaggle': None,
              'note': 'D4 TTA, chosen augmentation', **{k: m[k] for k in ('f1_clean', 'f1_shifted', 'f1_cigar_clean', 'f1_cigar_shifted', 'minutes')}}
             for m in fold_metrics]
    rows += [{'date': now, 'fold': 'OOF', 'train_set': 'clean (keep=True)', 'experiment': RUN, 'kaggle': None,
              'note': f'OOF, TTA {n}' + (' (submission.csv)' if n == chosen else ''),
              **{k: r[k] for k in ('f1_clean', 'f1_shifted', 'f1_cigar_clean', 'f1_cigar_shifted')}} for n, r in res.items()]
    pd.DataFrame(rows).to_csv(run_dir / 'experiments_log.csv', index=False)
    json.dump({'run': RUN, 'model': MODEL, 'aug': AUG, 'tta_mode': chosen, 'folds': sorted(final_tags),
               'oof': {n: {k: v for k, v in r.items() if k.startswith('f1')} for n, r in res.items()},
               'epochs': EPOCHS, 'lr': LR, 'batch_size': BATCH_SIZE, 'img_size': IMG_SIZE, 'seed': SEED,
               'minutes_total': round((time.time() - T_START) / 60, 1),
               'device': torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else DEVICE.type},
              open(run_dir / f'{RUN}.json', 'w'), indent=2)
print(f'\ntotal time: {(time.time() - T_START) / 60:.0f} min | outputs in {run_dir}')

In [ ]:
import matplotlib.pyplot as plt
hist = [pd.read_csv(p) for p in sorted(run_dir.glob('*_history.csv'))]
if hist:
    fig, ax = plt.subplots(1, 3, figsize=(16, 3.8))
    for h in hist:
        tag = h.tag.iloc[0]
        ax[0].plot(h.epoch, h.train_loss, label=tag)
        e = h.dropna(subset=['f1_clean']) if 'f1_clean' in h else h.iloc[:0]
        ax[1].plot(e.epoch, e.f1_clean, 'o-', ms=3, label=tag)
        ax[2].plot(e.epoch, e.f1_shifted, 'o-', ms=3, label=tag)
    for a, t in zip(ax, ['train loss', 'F1 clean (EMA, no TTA)', 'F1 shifted (EMA, no TTA)']):
        a.set_title(t); a.set_xlabel('epoch'); a.grid(alpha=0.3); a.legend(fontsize=7)
    plt.tight_layout(); plt.show()